In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

# ── SLV Model Parameters ──
S0       = 100.0
sigma0   = 0.15
Y0       = 0.0
rho      = -0.50
gamma    = 0.50
kappa_mr = 1.0
T        = 1.0

# ── Simulation Grid ──
n_steps  = 100          # SLV simulation: fine grid (dt = 1/100)
dt       = T / n_steps
t_fine   = np.linspace(0, T, n_steps + 1)

# ── Exercise Dates (monthly, 12 dates) ──
n_exercise = 12
exercise_indices = np.round(np.linspace(0, n_steps, n_exercise + 1)).astype(int)
ts_exercise = t_fine[exercise_indices]   # shape (13,): t0, t1, ..., t12

# ── Calibration / Pricing Settings ──
N_cal     = 10_000      # paths for particle method
N_price   = 100_000     # paths for pricing
sigma_mkt = 0.15        # flat market implied vol
kappa_bw  = 1.0         # bandwidth tuning parameter

K = 100.0               # strike for American put
r = 0.0                 # risk-free rate (SLV notebook uses 0)
q = 0.0                 # dividend yield

In [ ]:
# ── Black-Scholes Pricer ──
def blackscholes_price(K, T, S, vol, r=0, q=0, callput='call'):
    F = S * np.exp((r - q) * T)
    w = vol**2 * T
    d1 = (np.log(F / K) + 0.5 * w) / np.sqrt(w)
    d2 = d1 - np.sqrt(w)
    opttype = 1 if callput.lower() == 'call' else -1
    price = opttype * (F * norm.cdf(opttype * d1) - K * norm.cdf(opttype * d2)) * np.exp(-r * T)
    return price

# ── Black-Scholes Implied Vol (scalar) ──
def blackscholes_impv_scalar(K, T, S, value, r=0, q=0, callput='call', tol=1e-6, maxiter=500):
    if (K <= 0) or (T <= 0):
        return np.nan
    F = S * np.exp((r - q) * T)
    K_norm = K / F
    value = value * np.exp(r * T) / F
    opttype = 1 if callput.lower() == 'call' else -1
    value -= max(opttype * (1 - K_norm), 0)
    if value < 0:
        return np.nan
    if value == 0:
        return 0
    j = 1
    p = np.log(K_norm)
    if K_norm >= 1:
        x0 = np.sqrt(2 * p)
        x1 = x0 - (0.5 - K_norm * norm.cdf(-x0) - value) * np.sqrt(2 * np.pi)
        while (abs(x0 - x1) > tol * np.sqrt(T)) and (j < maxiter):
            x0 = x1
            d1 = -p / x1 + 0.5 * x1
            x1 = x1 - (norm.cdf(d1) - K_norm * norm.cdf(d1 - x1) - value) * np.sqrt(2 * np.pi) * np.exp(0.5 * d1**2)
            j += 1
    else:
        x0 = np.sqrt(-2 * p)
        x1 = x0 - (0.5 * K_norm - norm.cdf(-x0) - value) * np.sqrt(2 * np.pi) / K_norm
        while (abs(x0 - x1) > tol * np.sqrt(T)) and (j < maxiter):
            x0 = x1
            d1 = -p / x1 + 0.5 * x1
            x1 = x1 - (K_norm * norm.cdf(x1 - d1) - norm.cdf(-d1) - value) * np.sqrt(2 * np.pi) * np.exp(0.5 * d1**2)
            j += 1
    return x1 / np.sqrt(T)

blackscholes_impv = np.vectorize(blackscholes_impv_scalar, excluded={'callput', 'tol', 'maxiter'})

# ── Quartic Kernel ──
def quartic_kernel(x):
    x = np.clip(x, -1, 1)
    return (x + 1)**2 * (1 - x)**2

In [3]:
# ── Helper Function Tests ──

# TEST 1: blackscholes_price (docstring example)
put_price = blackscholes_price(95, 0.25, 100, 0.2, r=0.05, callput='put')
print(f"BS Put Price: {put_price:.16f}")
print(f"Expected:     1.5342604771222823")
print(f"Match: {np.isclose(put_price, 1.5342604771222823)}\n")

# Put-call parity
call_p = blackscholes_price(100, 1.0, 100, 0.2, r=0.05, callput='call')
put_p  = blackscholes_price(100, 1.0, 100, 0.2, r=0.05, callput='put')
print(f"C - P = {call_p - put_p:.6f}")
print(f"S - Ke^(-rT) = {100 - 100*np.exp(-0.05):.6f}\n")

# TEST 2: blackscholes_impv roundtrip
for v in [0.10, 0.15, 0.20, 0.30]:
    cp = blackscholes_price(100, 1.0, 100, v, callput='call')
    iv = blackscholes_impv(100, 1.0, 100, cp, callput='call')
    print(f"Input vol={v:.2f} → price={cp:.6f} → recovered vol={iv:.6f} → error={abs(iv-v):.2e}")

# TEST 3: quartic_kernel
print(f"\nK(-1)={quartic_kernel(np.array([-1.0]))[0]:.4f}, K(0)={quartic_kernel(np.array([0.0]))[0]:.4f}, K(1)={quartic_kernel(np.array([1.0]))[0]:.4f}")
print(f"Symmetric: K(0.3)={quartic_kernel(np.array([0.3]))[0]:.6f}, K(-0.3)={quartic_kernel(np.array([-0.3]))[0]:.6f}")

BS Put Price: 1.5342604771222823
Expected:     1.5342604771222823
Match: True

C - P = 4.877058
S - Ke^(-rT) = 4.877058

Input vol=0.10 → price=3.987761 → recovered vol=0.100000 → error=8.33e-17
Input vol=0.15 → price=5.978529 → recovered vol=0.150000 → error=2.78e-16
Input vol=0.20 → price=7.965567 → recovered vol=0.200000 → error=2.78e-16
Input vol=0.30 → price=11.923538 → recovered vol=0.300000 → error=3.89e-16

K(-1)=0.0000, K(0)=1.0000, K(1)=0.0000
Symmetric: K(0.3)=0.828100, K(-0.3)=0.828100
